# 29. Hidden Signal Audit

28번 이후 점수 개선폭이 작아졌기 때문에, 큰 점프 가능성이 있는 숨은 신호를 Train OOF 안에서만 점검합니다.

- Test 값/분포/예측 분포는 사용하지 않습니다.
- `ID`, 원본 row order, modulo/bucket, 공원/거리 결측 패턴 등이 28 residual을 설명하는지 확인합니다.
- 이 노트북은 진단용이며 제출 파일을 생성하지 않습니다.


In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

# 28번을 재현해 현재 best line의 OOF prediction을 가져옵니다.
# 28 내부는 27/25/24/17을 leakage-safe 방식으로 재현합니다.
nb28 = json.loads((PROJECT_ROOT / 'notebooks' / '28_finer_refined_comparable.ipynb').read_text())
code28 = ''.join(nb28['cells'][1]['source'])
ns28 = {}
exec(code28, ns28)

rmse = ns28['rmse']
train = ns28['train'].copy()
oof = ns28['oof'].copy()
folds = ns28['folds']
inner15 = ns28['inner15']
pred17_all = ns28['pred17_all']
pred28_all = ns28['pred_tables'][ns28['best_key']]

# OOF row와 원본 Train row metadata를 맞춥니다.
meta_parts = []
train_with_row = train.copy()
train_with_row['Original_Row'] = np.arange(len(train_with_row))
for fold, (_, valid_months) in enumerate(inner15['TIME_FOLDS'], start=1):
    valid_meta = train_with_row.loc[train_with_row['Transaction_YearMonth'].isin(valid_months)].copy()
    valid_meta['Fold'] = fold
    meta_parts.append(valid_meta.reset_index(drop=True))
meta = pd.concat(meta_parts, ignore_index=True)
assert len(meta) == len(oof)

audit = oof.copy().reset_index(drop=True)
for col in ['ID', 'Original_Row', 'Transaction_YearMonth', 'Nearby_Parks', 'Distance_to_Subway']:
    audit[col] = meta[col].to_numpy()
audit['Pred28'] = pred28_all
audit['Residual28'] = audit['Target'] - audit['Pred28']
audit['ID_Num'] = audit['ID'].str.extract(r'(\d+)').astype(int)[0]
audit['Transaction_Month'] = audit['Transaction_YearMonth'] % 100
audit['Distance_Missing'] = audit['Distance_to_Subway'].isna().astype(int)

base_rmse = rmse(audit['Target'], audit['Pred28'])
print(f'28 pooled OOF RMSE: {base_rmse:,.6f}')
display(audit[['Fold', 'ID', 'Original_Row', 'Transaction_YearMonth', 'Target', 'Pred28', 'Residual28']].head())

# 1) 단순 상관/분포 점검: Train OOF residual과 ID/row가 연관되는지 확인합니다.
corr_rows = []
numeric_probe_cols = ['ID_Num', 'Original_Row', 'Transaction_YearMonth', 'Transaction_Month', 'Nearby_Parks', 'Distance_to_Subway', 'Distance_Missing']
for fold in folds:
    frame = audit.loc[audit['Fold'] == fold]
    for col in numeric_probe_cols:
        x = frame[col]
        if x.notna().sum() < 5 or x.nunique(dropna=True) <= 1:
            corr = np.nan
        else:
            corr = np.corrcoef(x.fillna(x.median()), frame['Residual28'])[0, 1]
        corr_rows.append({'Fold': fold, 'Feature': col, 'Residual_Corr': corr})
corr_table = pd.DataFrame(corr_rows)
display(corr_table.pivot(index='Feature', columns='Fold', values='Residual_Corr'))

# 2) 고정 bucket/modulo feature를 만듭니다. qcut처럼 전체 분포를 fit하지 않고 고정 규칙만 사용합니다.
def add_hidden_signal_groups(df):
    x = df.copy()
    for mod in [2, 3, 5, 7, 10, 20]:
        x[f'ID_Mod_{mod}'] = (x['ID_Num'] % mod).astype(str)
        x[f'Row_Mod_{mod}'] = (x['Original_Row'] % mod).astype(str)
    for width in [50, 100, 200, 300, 500]:
        x[f'ID_Bin_{width}'] = (x['ID_Num'] // width).astype(str)
        x[f'Row_Bin_{width}'] = (x['Original_Row'] // width).astype(str)
    x['ID_Last2'] = (x['ID_Num'] % 100).astype(str)
    x['ID_Last3'] = (x['ID_Num'] % 1000).astype(str)
    x['Month_Str'] = x['Transaction_Month'].astype(str)
    x['Nearby_Parks_Str'] = x['Nearby_Parks'].astype(str)
    x['Distance_Missing_Str'] = x['Distance_Missing'].astype(str)
    x['Gu_ID_Mod10'] = x['Gu'].astype(str) + '_' + x['ID_Mod_10']
    x['Dong_ID_Mod10'] = x['Dong'].astype(str) + '_' + x['ID_Mod_10']
    x['Gu_Row_Bin100'] = x['Gu'].astype(str) + '_' + x['Row_Bin_100']
    return x

audit = add_hidden_signal_groups(audit)

def smoothed_residual(fit_frame, pred_frame, group_col, prior):
    global_mean = fit_frame['Residual28'].mean()
    stats = fit_frame.groupby(group_col)['Residual28'].agg(['sum', 'count'])
    mapping = (stats['sum'] + prior * global_mean) / (stats['count'] + prior)
    return pred_frame[group_col].map(mapping).fillna(global_mean).to_numpy()

candidate_groups = [
    'ID_Mod_2', 'ID_Mod_3', 'ID_Mod_5', 'ID_Mod_7', 'ID_Mod_10', 'ID_Mod_20',
    'Row_Mod_2', 'Row_Mod_3', 'Row_Mod_5', 'Row_Mod_7', 'Row_Mod_10', 'Row_Mod_20',
    'ID_Bin_50', 'ID_Bin_100', 'ID_Bin_200', 'ID_Bin_300', 'ID_Bin_500',
    'Row_Bin_50', 'Row_Bin_100', 'Row_Bin_200', 'Row_Bin_300', 'Row_Bin_500',
    'ID_Last2', 'ID_Last3', 'Month_Str', 'Nearby_Parks_Str', 'Distance_Missing_Str',
    'Gu_ID_Mod10', 'Dong_ID_Mod10', 'Gu_Row_Bin100',
]
priors = [10, 30, 50, 100]
alphas = [0.02, 0.05, 0.08, 0.10, 0.15]

rows = []
detail_tables = {}
for group_col in candidate_groups:
    for prior in priors:
        for alpha in alphas:
            pred_parts = []
            detail = []
            for fold in folds:
                valid_frame = audit.loc[audit['Fold'] == fold].copy()
                y = valid_frame['Target'].to_numpy()
                pred28 = valid_frame['Pred28'].to_numpy()
                if fold == 1:
                    pred29 = pred28.copy()
                    mean_abs_move = 0.0
                else:
                    fit_frame = audit.loc[audit['Fold'] < fold].copy()
                    adj = smoothed_residual(fit_frame, valid_frame, group_col, prior)
                    pred29 = np.clip(pred28 + alpha * adj, 0, None)
                    mean_abs_move = float(np.mean(np.abs(pred29 - pred28)))
                pred_parts.append(pred29)
                detail.append({
                    'Fold': fold,
                    'Rows': len(valid_frame),
                    '28_RMSE': rmse(y, pred28),
                    '29_RMSE': rmse(y, pred29),
                    '29_vs_28_Improvement': rmse(y, pred28) - rmse(y, pred29),
                    'Mean_abs_move': mean_abs_move,
                    'Group_Unique': int(valid_frame[group_col].nunique()),
                })
            pred_all = np.concatenate(pred_parts)
            detail = pd.DataFrame(detail)
            eligible = detail.loc[detail['Fold'] > 1]
            key = (group_col, prior, alpha)
            detail_tables[key] = detail
            rows.append({
                'Group': group_col,
                'Prior': prior,
                'Alpha': alpha,
                '28_Pooled_RMSE': base_rmse,
                '29_Pooled_RMSE': rmse(audit['Target'], pred_all),
                'Pooled_Improvement': base_rmse - rmse(audit['Target'], pred_all),
                'Improved_Eligible_Folds': int((eligible['29_vs_28_Improvement'] > 0).sum()),
                'Worst_Eligible_Improvement': eligible['29_vs_28_Improvement'].min(),
                'Fold2_Improvement': detail.loc[detail['Fold'] == 2, '29_vs_28_Improvement'].iloc[0],
                'Fold3_Improvement': detail.loc[detail['Fold'] == 3, '29_vs_28_Improvement'].iloc[0],
                'Mean_abs_move': detail['Mean_abs_move'].mean(),
            })

group_summary = pd.DataFrame(rows).sort_values(
    ['Pooled_Improvement', 'Worst_Eligible_Improvement'], ascending=[False, False]
).reset_index(drop=True)
display(group_summary.head(40))

stable_group = group_summary.query(
    'Pooled_Improvement > 0.30 and Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 0.05 and Mean_abs_move < 40'
).copy()
if len(stable_group) > 0:
    print('Train OOF에서 안정적으로 residual을 설명하는 hidden group 후보가 있습니다:')
    display(stable_group.head(10))
    best_group_key = (stable_group.iloc[0]['Group'], int(stable_group.iloc[0]['Prior']), float(stable_group.iloc[0]['Alpha']))
    display(detail_tables[best_group_key])
else:
    print('ID/row-order group residual correction 중 안정 조건을 통과한 후보가 없습니다.')

# 3) ID/row-order numeric feature로 Residual28을 예측하는 간단한 OOF residual model 진단.
# 이 역시 이전 OOF fold만 학습합니다. 제출에는 사용하지 않습니다.
model_features = [
    'ID_Num', 'Original_Row', 'Transaction_YearMonth', 'Transaction_Month',
    'Nearby_Parks', 'Distance_to_Subway', 'Distance_Missing',
    'Gu', 'Dong', 'ID_Mod_10', 'Row_Mod_10', 'ID_Bin_100', 'Row_Bin_100',
]
cat_cols = ['Gu', 'Dong', 'ID_Mod_10', 'Row_Mod_10', 'ID_Bin_100', 'Row_Bin_100']
num_cols = [c for c in model_features if c not in cat_cols]

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_hidden_model():
    preprocessor = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='__missing__')),
            ('onehot', encoder),
        ]), cat_cols),
    ])
    return Pipeline([
        ('prep', preprocessor),
        ('model', HistGradientBoostingRegressor(
            max_iter=250, learning_rate=0.03, max_leaf_nodes=10,
            min_samples_leaf=25, l2_regularization=50.0, random_state=2029,
        )),
    ])

model_rows = []
for alpha in [0.02, 0.05, 0.08, 0.10, 0.15]:
    pred_parts = []
    detail = []
    for fold in folds:
        valid_frame = audit.loc[audit['Fold'] == fold].copy()
        y = valid_frame['Target'].to_numpy()
        pred28 = valid_frame['Pred28'].to_numpy()
        if fold == 1:
            pred29 = pred28.copy()
            move = 0.0
        else:
            fit_frame = audit.loc[audit['Fold'] < fold].copy()
            pipe = make_hidden_model()
            pipe.fit(fit_frame[model_features], fit_frame['Residual28'])
            adj = pipe.predict(valid_frame[model_features])
            lo, hi = np.percentile(fit_frame['Residual28'], [10, 90])
            adj = np.clip(adj, lo, hi)
            pred29 = np.clip(pred28 + alpha * adj, 0, None)
            move = float(np.mean(np.abs(pred29 - pred28)))
        pred_parts.append(pred29)
        detail.append({
            'Fold': fold,
            '28_RMSE': rmse(y, pred28),
            '29_Model_RMSE': rmse(y, pred29),
            'Model_vs_28_Improvement': rmse(y, pred28) - rmse(y, pred29),
            'Mean_abs_move': move,
        })
    pred_all = np.concatenate(pred_parts)
    detail = pd.DataFrame(detail)
    eligible = detail.loc[detail['Fold'] > 1]
    model_rows.append({
        'Alpha': alpha,
        'Pooled_Improvement': base_rmse - rmse(audit['Target'], pred_all),
        'Improved_Eligible_Folds': int((eligible['Model_vs_28_Improvement'] > 0).sum()),
        'Worst_Eligible_Improvement': eligible['Model_vs_28_Improvement'].min(),
        'Fold2_Improvement': detail.loc[detail['Fold'] == 2, 'Model_vs_28_Improvement'].iloc[0],
        'Fold3_Improvement': detail.loc[detail['Fold'] == 3, 'Model_vs_28_Improvement'].iloc[0],
        'Mean_abs_move': detail['Mean_abs_move'].mean(),
    })
hidden_model_summary = pd.DataFrame(model_rows).sort_values('Pooled_Improvement', ascending=False)
display(hidden_model_summary)

leakage_audit = pd.Series({
    '28 baseline reproduced from leakage-safe notebook': True,
    'All hidden-signal checks use Train OOF only': True,
    'Group residual maps fitted only on earlier OOF folds': True,
    'Hidden residual model fitted only on earlier OOF folds': True,
    'No Test values/distribution/statistics used for audit decisions': True,
    'No submission file generated by this diagnostic notebook': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

result = {
    'create_submission': False,
    'best_group_improvement': float(group_summary['Pooled_Improvement'].max()),
    'stable_group_candidates': int(len(stable_group)),
    'best_hidden_model_improvement': float(hidden_model_summary['Pooled_Improvement'].max()),
}
print(result)


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/09_residual08_cluster_correction_submission.csv


...

,Group,Prior,Alpha,28_Pooled_RMSE,29_Pooled_RMSE,Pooled_Improvement,Improved_Eligible_Folds,Worst_Eligible_Improvement,Fold2_Improvement,Fold3_Improvement,Mean_abs_move
0,Row_Mod_7,10,0.150000,"2,396.213670","2,395.420523",0.793147,2,0.188853,0.188853,2.174558,15.415937
1,Row_Mod_7,30,0.150000,"2,396.213670","2,395.616233",0.597437,2,0.340688,0.340688,1.434887,13.123981
2,Row_Mod_7,10,0.100000,"2,396.213670","2,395.662642",0.551028,2,0.148997,0.148997,1.492422,10.277291
4,Row_Mod_7,50,0.150000,"2,396.213670","2,395.755700",0.457970,2,0.420034,0.420034,0.937404,11.789330
5,Row_Mod_7,10,0.080000,"2,396.213670","2,395.765725",0.447946,2,0.126588,0.126588,1.207602,8.221833
6,Row_Mod_7,30,0.100000,"2,396.213670","2,395.799306",0.414365,2,0.242603,0.242603,0.988695,8.749321
8,ID_Bin_300,10,0.150000,"2,396.213670","2,395.841296",0.372374,2,0.205714,0.205714,0.901078,22.493294
10,Row_Mod_7,30,0.080000,"2,396.213670","2,395.877036",0.336635,2,0.199035,0.199035,0.801227,6.999457
11,Row_Mod_7,50,0.100000,"2,396.213670","2,395.895289",0.318382,2,0.292462,0.292462,0.651204,7.859553
12,ID_Bin_300,10,0.100000,"2,396.213670","2,395.909368",0.304302,2,0.197600,0.197600,0.706184,14.995529


,Fold,Rows,28_RMSE,29_RMSE,29_vs_28_Improvement,Mean_abs_move,Group_Unique
0,1,278,"2,107.743326","2,107.743326",0.000000,0.000000,7
1,2,244,"2,583.858593","2,583.669740",0.188853,18.728843,7
2,3,246,"2,506.635060","2,504.460501",2.174558,27.518967,7


,Alpha,Pooled_Improvement,Improved_Eligible_Folds,Worst_Eligible_Improvement,Fold2_Improvement,Fold3_Improvement,Mean_abs_move
0,0.020000,-0.036322,1,-0.597230,-0.597230,0.502344,6.607515
1,0.050000,-0.164921,1,-1.606125,-1.606125,1.150710,16.518787
2,0.080000,-0.382445,1,-2.750539,-2.750539,1.672810,26.430060
3,0.100000,-0.576853,1,-3.588681,-3.588681,1.950690,33.037575
4,0.150000,-1.235671,1,-5.946804,-5.946804,2.399602,49.556362


28 baseline reproduced from leakage-safe notebook                  True
All hidden-signal checks use Train OOF only                        True
Group residual maps fitted only on earlier OOF folds               True
Hidden residual model fitted only on earlier OOF folds             True
No Test values/distribution/statistics used for audit decisions    True
No submission file generated by this diagnostic notebook           True
Name: Passed, dtype: bool

{'create_submission': False, 'best_group_improvement': 0.7931473671778804, 'stable_group_candidates': 10, 'best_hidden_model_improvement': -0.0363218617030725}
